# Clase 6 — RAG con un archivo de texto: el asistente del Circo Aurora

En la Clase 6 el agente ya consultaba una base de conocimiento en JSON. Hoy damos el paso siguiente: **Recuperación Aumentada Generativa (RAG)** sobre un documento de texto plano, como los que existen en cualquier negocio real. Nuestro caso: un circo que recibe consultas de potenciales clientes sobre funciones, horarios y reglas.

> La recuperación aporta los hechos. La generación los redacta. Si el archivo no contiene la respuesta, el agente debe saber callar.

## Objetivos

- Explicar qué es un sistema RAG y nombrar sus etapas.
- Ubicar la recuperación dentro de la arquitectura general de un agente.
- Trocear un archivo de texto en fragmentos recuperables.
- Conectar la evidencia recuperada a un LLM local real.
- Mejorar el desempeño del agente **editando el documento**, sin tocar el código.

## Agenda

| Bloque | Tiempo | Resultado |
|---|---|---|
| Explicación guiada — qué es RAG | 30 minutos | Diagrama y pipeline entendidos |
| Ejercicio — el asistente del circo | 40 minutos | Documento mejorado y evaluación medida |

---
# Parte 1 — Explicación guiada (30 minutos)

## ¿Qué problema resuelve RAG?

Un LLM es un gran redactor con dos limitaciones conocidas:

1. **No sabe** lo que no está en sus datos de entrenamiento: el horario de nuestro circo, las reglas de nuestra carpa, los precios de esta temporada.
2. **Igual responde**: ante un hueco de conocimiento produce un texto plausible, convencido y falso. Eso es una alucinación.

RAG ataca el problema por el lado de los datos: antes de generar, **buscamos en una fuente propia** los fragmentos relevantes y los entregamos al modelo como evidencia. El modelo ya no recuerda: **lee y redacta**.

    pregunta → recuperar fragmentos → generar respuesta fundamentada
                     ↓
             si no hay evidencia
                     ↓
             responder "no sé" y escalar

## Las tres etapas de un sistema RAG

| Etapa | Qué hace | En esta clase |
|---|---|---|
| **Indexar** | Preparar la fuente: trocear el documento en fragmentos | Troceamos el `.txt` por secciones |
| **Recuperar** | Ante una pregunta, elegir los fragmentos más relevantes | Búsqueda literal por palabras compartidas |
| **Generar** | Redactar la respuesta usando solo esa evidencia | LLM local con un prompt restringido |

El nombre lo describe completo: **Generativa** (hay un LLM que escribe) **Aumentada** (se aumenta el prompt con evidencia) **Recuperación** (la evidencia viene de una búsqueda sobre una base propia).

## RAG dentro de la arquitectura de un agente

En la Clase 4 vimos que un agente es un LLM con herramientas controladas por Python. RAG no reemplaza esa arquitectura: **es una herramienta más**. El agente decide cuándo consultarla, igual que decidía usar una calculadora o un clasificador.

~~~mermaid
flowchart LR
    U[Usuario pregunta] --> A[Agente]
    A -->|elige herramienta| R[Recuperador]
    R --> B[(circo_aurora.txt)]
    R -->|fragmentos| G[LLM genera respuesta]
    A --> G --> U
    R -.->|sin evidencia| E[Escalar a humano]
~~~

Dos decisiones de diseño importantes:

- **La fuente es editable por gente no programadora.** Quien administra el circo puede corregir un horario en el `.txt` y el agente mejora sin reescribir código ni reentrenar nada.
- **El recuperador puede negarse.** Con un umbral de similitud, el sistema distingue "no hay información" de "hay información pero es mala". Negarse es preferible a inventar.

## Del JSON estructurado al texto plano

La base de la Clase 5 era una lista de documentos con campos. El mundo real suele dar algo menos amable: un manual, un reglamento, un README. Texto corrido, sin etiquetas.

Nuestro `circo_aurora.txt` está en el punto medio: texto plano, pero con secciones marcadas con `###`. Ese marcador nos permite **trocear** el documento en fragmentos, que cumplen el mismo rol que los documentos KB01–KB11 de la clase anterior.

## Paso 1 — Leer el documento

Cargamos el `.txt` desde la misma carpeta donde está el notebook. Descargá `circo_aurora.txt` junto a este `.ipynb` y va a funcionar en cualquier ubicación.

In [ ]:
from pathlib import Path

ruta_doc=Path("circo_aurora.txt")
documento=ruta_doc.read_text(encoding="utf-8")
print("Leído desde:",ruta_doc)
print(documento[:300])

**Leamos las decisiones.** El notebook busca el archivo en su **propia carpeta**, sin importar dónde esté descargado: en tu compu, en Google Colab o en otra estructura de carpetas. La única regla es que `circo_aurora.txt` esté al lado del `.ipynb`. Usamos `Path` de `pathlib` porque los strings no saben leer archivos. Leemos el documento entero como una sola cadena; qué fragmento sirve para cada pregunta lo resuelve el troceado.

## Paso 2 — Trocear el documento (indexación)

Cada línea que empieza con `###` abre una sección. El troceador corta ahí y produce un fragmento por sección, con su identificador.

In [ ]:
def trocear_texto(documento):
    fragmentos=[]
    actual=None
    for linea in documento.splitlines():
        if linea.startswith("###"):
            if actual: fragmentos.append(actual)
            titulo=linea.replace("###","").strip()
            actual={"id":f"FR{len(fragmentos)+1:02d}",
                    "titulo":titulo,"contenido":""}
        elif actual is not None:
            actual["contenido"]+=linea.strip()+" "
    if actual: fragmentos.append(actual)
    for f in fragmentos:
        f["contenido"]=f["contenido"].strip()
    return fragmentos

fragmentos=trocear_texto(documento)
for f in fragmentos:
    print(f["id"],"—",f["titulo"],"|",f["contenido"][:60],"...")

**Leamos las decisiones.** El marcador `###` convierte un archivo opaco en una mini-base de datos. Noten que `FR00` (el encabezado del archivo) no entra: no empieza con `###` y está antes de la primera sección, así que el troceador lo descarta. Si mañana el circo agrega una sección "Preguntas frecuentes", aparece sola como fragmento nuevo.

## Paso 3 — Recuperación literal

Contamos palabras significativas compartidas entre pregunta y fragmento. Sin librerías de ML, para ver el mecanismo con claridad.

In [ ]:
import unicodedata

PALABRAS_VACIAS={"el","la","los","las","de","del","un","una","y","o",
                  "que","cómo","como","cual","cuál","mi","me","es","son",
                  "para","por","con","en","a","hay","puedo","cuándo",
                  "cuando","qué"}

def raiz(palabra):
    sin_tilde="".join(c for c in unicodedata.normalize("NFD",palabra)
                      if unicodedata.category(c)!="Mn")
    return sin_tilde[:3]

def palabras(texto):
    limpias="".join(c.lower() if c.isalnum() else " " for c in texto)
    brutas=[p for p in limpias.split()
            if len(p)>2 and p not in PALABRAS_VACIAS]
    return {raiz(p) for p in brutas}

def buscar_fragmentos(pregunta,k=2):
    consulta=palabras(pregunta)
    resultados=[]
    for f in fragmentos:
        vocab=palabras(f["titulo"]+" "+f["contenido"])
        interseccion=consulta & vocab
        resultados.append({**f,"puntaje":len(interseccion),
                           "coincidencias":sorted(interseccion)})
    return sorted(resultados,key=lambda x:x["puntaje"],reverse=True)[:k]

buscar_fragmentos("¿A qué hora es la función del sábado?")

**Leamos las decisiones.** La columna `coincidencias` hace auditable el porqué de cada resultado. El truco de la raíz: cortamos cada palabra a 3 letras y quitamos tildes, así "sábados" y "sábado" se vuelven `sab` y "bebé"/"bebidas" comparten `beb`. Es un stemming casero, imperfecto pero suficiente, y sin dependencias. Igual tiene un límite que verán en el ejercicio: si la pregunta usa una palabra y el fragmento otra distinta ("horario" vs "funciones"), la búsqueda literal no une nada.

## Paso 4 — Umbral: el recuperador puede negarse

El mejor fragmento siempre existe. El umbral decide si es evidencia suficiente.

El **umbral** (`umbral=2`) es el puntaje mínimo de palabras compartidas que exige el recuperador antes de aceptar un fragmento como evidencia. Es la línea que separa *"tengo información"* de *"no tengo nada que decir"*.

##### El problema que resuelve

`buscar_fragmentos()` **siempre devuelve algo**: ordena los 5 fragmentos del mejor al peor y te da el primero. Hasta una pregunta absurda como *"¿Venden pizza en el circo?"* tiene un "mejor resultado" — probablemente FR04 (Reglas) con 1 raíz compartida. Pero 1 palabra suelta no es evidencia, es ruido.

```python
def recuperar_evidencia(pregunta,umbral=2):
    ...
    if mejor["puntaje"]<umbral:          # ← la decisión
        return {"encontrado":False, ...}
```

| Pregunta | Mejor fragmento | Puntaje | Umbral 2 | Veredicto |
|---|---|---|---|---|
| ¿A qué hora es la función del sábado? | FR02 Horarios | 3 | 3 ≥ 2 | ✅ Evidencia |
| ¿Se puede entrar con alimentos? | FR04 Reglas | 2 | 2 ≥ 2 | ✅ Evidencia |
| ¿Venden pizza en el circo? | (cualquiera) | 1 | 1 < 2 | ❌ Se niega |

#### Por qué no se inventa ese número

El umbral es un **hiperparámetro**: no sale de ninguna fórmula, se elige probando. En el ejercicio de la Clase 6 (versión TF-IDF) los alumnos barían 0.05/0.12/0.25/0.50 y veían el trade-off:

- **Umbral muy bajo** → acepta casi todo → el LLM redacta sobre evidencia débil → respuestas sin fundamento (más alucinación).
- **Umbral muy alto** → rechaza casi todo → el agente se niega aunque la respuesta exista en el archivo (agente inútil).

Acá con puntajes enteros, `2` funciona porque un solo match de 3 letras es demasiado casual, pero dos ya sugieren tema común.



In [ ]:
def recuperar_evidencia(pregunta,umbral=2):
    resultados=buscar_fragmentos(pregunta,1)
    mejor=resultados[0]
    if mejor["puntaje"]<umbral:
        return {"encontrado":False,
                "motivo":"sin palabras en común suficientes",
                "puntaje":mejor["puntaje"]}
    return {"encontrado":True,"fragmento_id":mejor["id"],
            "titulo":mejor["titulo"],"contenido":mejor["contenido"],
            "puntaje":mejor["puntaje"]}

for q in ["¿A qué hora es la función del sábado?",
          "¿Se puede entrar con alimentos?",
          "¿Venden pizza en el circo?"]:
    r=recuperar_evidencia(q)
    destino=r.get("fragmento_id",r.get("motivo","?"))
    print(q,"→",destino,"| puntaje:",r["puntaje"])

## Paso 5 — El LLM local real

El modelo descargado es `LFM2.5-1.2B-Instruct` en formato GGUF. Si no está disponible, el notebook sigue funcionando en modo extractivo.

In [ ]:
import time

USAR_MODELO=True
llm=None
try:
    from llama_cpp import Llama
    from huggingface_hub import hf_hub_download
    ruta_modelo=hf_hub_download(
        repo_id="unsloth/LFM2.5-1.2B-Instruct-GGUF",
        filename="LFM2.5-1.2B-Instruct-Q8_0.gguf")
    llm=Llama(model_path=ruta_modelo,
              n_ctx=4096,
              n_gpu_layers=0,
              verbose=False)
    print("Modelo cargado.")
except Exception as error:
    USAR_MODELO=False
    print("Sin modelo local:",error)

def consultar_llm(pregunta,evidencia=None):
    if evidencia is None:
        system="Sos el asistente del Circo Aurora."
    else:
        system=("Sos el asistente del Circo Aurora. Respondé usando "
                "únicamente la EVIDENCIA. Si no alcanza, decí que "
                "consulte en boletería. Cerrá con FUENTE: "
                +evidencia["fragmento_id"]+".\nEVIDENCIA: "
                +evidencia["contenido"])
    if not USAR_MODELO:
        return {"texto":evidencia["contenido"] if evidencia
                else "No encuentro evidencia suficiente.",
                "modo":"extractivo"}
    inicio=time.time()
    salida=llm.create_chat_completion(
        messages=[{"role":"system","content":system},
                  {"role":"user","content":pregunta}],
        temperature=0,max_tokens=200)
    return {"texto":salida["choices"][0]["message"]["content"],
            "modo":"modelo local",
            "duracion_s":round(time.time()-inicio,1)}

**Leamos las decisiones.** Tres detalles importan:

- `temperature=0`: para un asistente factual queremos la redacción más determinista posible.
- La instrucción del sistema **prohíbe completar con conocimiento general**: el modelo solo puede usar la evidencia.
- Si no hay modelo, `consultar_llm` degrada a respuesta extractiva (devuelve la evidencia tal cual). El pipeline nunca se rompe.

Probemos el contraste clave: el mismo modelo, **con y sin** recuperación.

In [ ]:
pregunta="¿A qué hora abre la puerta el sábado?"

sin_rag=consultar_llm(pregunta)
con_rag=consultar_llm(pregunta,recuperar_evidencia(pregunta))

print("SIN RAG :",sin_rag["texto"][:200])
print("CON RAG :",con_rag["texto"][:200])

El modelo sin evidencia suele producir un horario genérico o inventado. Con evidencia, repite el dato del archivo. **Esa es toda la idea de RAG**: mover la verdad del modelo al documento.

## Paso 6 — El agente RAG completo

Unimos las etapas con una traza observable, como venimos haciendo desde la Clase 2.

In [ ]:
def agente_rag(pregunta,umbral=2):
    evidencia=recuperar_evidencia(pregunta,umbral)
    if not evidencia["encontrado"]:
        return {"respuesta":"No tengo ese dato en mi información "
                             "del circo. Consultá en boletería.",
                "fuente":None,"requiere_revision":True,
                "traza":[{"paso":"recuperar","resultado":evidencia}]}
    salida=consultar_llm(pregunta,evidencia)
    fuente_presente=evidencia["fragmento_id"] in salida["texto"]
    return {"respuesta":salida["texto"],
            "fuente":evidencia["fragmento_id"],
            "requiere_revision":not fuente_presente,
            "traza":[{"paso":"recuperar","resultado":evidencia},
                     {"paso":"generar","modo":salida["modo"]}]}

# pregunta de ejemplo
resultado=agente_rag("¿Se puede entrar con comida?")

print(resultado["respuesta"],"\n")
print("Fuente:",resultado["fuente"],
      "| Requiere revisión:",resultado["requiere_revision"])

---
# Parte 2 — Ejercicio: el asistente del circo (40 minutos)

## La consigna

La dirección del Circo Aurora probó el asistente con clientes reales y **falla en varias consultas**. Ustedes son el equipo de datos: deben mejorar el desempeño del agente **editando únicamente el archivo `circo_aurora.txt`**. El código no se toca.

Es RAG en su forma más pura: la calidad del asistente depende de la calidad de la fuente.

~~~mermaid
flowchart LR
    A[1. Diagnosticar] --> B[2. Editar el txt]
    B --> C[3. Re-evaluar]
    C -->|sigue fallando| B
    C -->|todo pasa| D[Entrega]
~~~

Reiniciá el kernel **después de cada edición del archivo** (o volvé a ejecutar las celdas de carga y troceado) para que el cambio se vea reflejado.

### Tarea 1 — Diagnosticar (10 minutos)

Ejecutá la evaluación y anotá qué preguntas fallan y por qué. Clasificá cada fallo: ¿el dato no está en el archivo? ¿está pero redactado con otras palabras? ¿está enterrado en una sección muy larga?

In [ ]:
import pandas as pd

preguntas_prueba=[
 "¿Qué día no hay funciones?",            # esperado: FR02
 "¿Cuánto sale la entrada para un bebé?", # esperado: FR03
 "¿Puedo usar dron para filmar?",         # esperado: FR04
 "¿Dónde queda la carpa?",                # esperado: FR05
 "¿Cuánto dura el show?",                 # esperado: FR01
 "¿Cómo llego en transporte público?",    # esperado: FR05
 "¿Venden bebidas adentro?",              # esperado: rechazo o FR04
]

def evaluar(preguntas):
    filas=[]
    for pregunta in preguntas:
        r=agente_rag(pregunta)
        filas.append({"pregunta":pregunta,
                      "fuente":r["fuente"],
                      "requiere_revision":r["requiere_revision"],
                      "respuesta":r["respuesta"][:80]})
    return pd.DataFrame(filas)

evaluar(preguntas_prueba)

### Tarea 2 — Editar el documento (20 minutos)

Abrí `modulo_4/datos/circo_aurora.txt` y mejorá la fuente. Estrategias, de menor a mayor esfuerzo:

1. **Redactar con las palabras del cliente.** Si preguntan "bebé" y el archivo dice "menores de 2 años", agregá la palabra del cliente. La búsqueda literal solo une vocabularios idénticos.
2. **Agregar secciones nuevas** con `###` para datos que faltan (¿estacionamiento? ¿accesibilidad? decidilo vos).
3. **Partir secciones muy largas** en dos: fragmentos cortos y temáticos recuperan mejor.

Cada vez que edites, volvé a correr la carga y la evaluación.

### Tarea 3 — Medir la mejora (10 minutos)

Corré la evaluación final, compará con la del diagnóstico y respondé las preguntas de entrega.

In [ ]:
# TODO: volvé a correr esta celda después de editar el txt
evaluar(preguntas_prueba)

### Entrega

Responder brevemente:

1. ¿Qué preguntas fallaban en el diagnóstico y a qué tipo de fallo correspondían?
2. ¿Qué cambios al `.txt` resolvieron cada fallo? Cita el fragmento editado.
3. ¿Hubo algún fallo que **no** pudiste arreglar editando el documento? ¿Por qué?
4. ¿Qué riesgo corrés si para "arreglar" el agente escribís en el txt algo que no es verdad?

**Criterio de logro:** al menos 6 de las 8 verificaciones del test pasan, y sabés explicar qué cambio del documento produjo cada mejora.

In [ ]:
# Test de verificación: corrélo al final del ejercicio.
# Evalúa la RECUPERACIÓN (qué fragmento usa el agente), no la redacción.
def test_final():
    esperados=["FR02","FR03","FR04","FR05","FR01","FR05"]
    aprobadas=0
    for pregunta,esperado in zip(preguntas_prueba[:6],esperados):
        r=agente_rag(pregunta)
        ok=r["fuente"]==esperado
        aprobadas+=ok
        print(("✔" if ok else "✘"),pregunta,"→",r["fuente"])
    r=agente_rag(preguntas_prueba[6])
    ok=r["fuente"] in (None,"FR04")
    aprobadas+=ok
    print(("✔" if ok else "✘"),preguntas_prueba[6],"→",r["fuente"])
    print(f"\nResultado: {aprobadas}/8")

test_final()

---
## ✅ Cierre

1. **RAG** = recuperar evidencia de una fuente propia antes de generar con el LLM.
2. Un sistema RAG tiene tres etapas: **indexar, recuperar, generar**. En el agente, la recuperación es una herramienta más, controlada por Python.
3. Un documento de texto con secciones marcadas se trocea en fragmentos y se comporta como una base de conocimiento.
4. El recuperador con umbral puede **negarse a responder**, y el prompt restringido evita que el modelo complete huecos.
5. La gran ventaja práctica: **se mejora el asistente editando el documento**, no el código ni el modelo.

> Un RAG es tan bueno como su fuente. La ingeniería no está en el modelo: está en el documento.

En la Clase 7 incorporaremos imágenes como entrada del agente.